<a href="https://colab.research.google.com/github/ProfAndersonVanin/IBM3130-PLN-2026/blob/main/semana-04/aula_bow_tfidf_nltk_spacy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 Aula: Representação de Texto — Bag of Words (BoW) & TF-IDF

Nesta aula prática no **Google Colab**, você aprenderá os conceitos fundamentais de vetorização de texto em Processamento de Linguagem Natural (NLP) e como implementá-los usando as bibliotecas **NLTK**, **spaCy** e **scikit-learn**.

---
## 1. Fundamentos Teóricos

Modelos de Machine Learning trabalham com números, não com palavras diretamente. Por isso, precisamos transformar textos em vetores numéricos.

### 1.1 Bag of Words (BoW)
- **Conceito**: Cria um vocabulário com todas as palavras únicas do corpus. Cada documento é representado por um vetor onde cada posição corresponde à contagem de vezes que uma palavra específica ocorreu.
- **Premissa**: A ordem e a gramática são ignoradas (por isso o nome *"saco de palavras"*).
- **Limitação**: Palavras muito frequentes (como preposições e artigos) dominam a representação, embora carreguem pouco significado semântico.

### 1.2 TF-IDF (Term Frequency – Inverse Document Frequency)
O TF-IDF balanceia a frequência de um termo no documento com a sua raridade em todo o corpus:

$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$$

1. **TF (Term Frequency)**: Mede a frequência local do termo $t$ no documento $d$:
   $$\text{TF}(t, d) = \frac{\text{Contagem de } t \text{ em } d}{\text{Total de termos em } d}$$
2. **IDF (Inverse Document Frequency)**: Penaliza termos que aparecem em quase todos os documentos do corpus $D$ ($N$ documentos no total):
   $$\text{IDF}(t, D) = \log\left(\frac{N}{|\{d \in D : t \in d\}|}\right)$$

---
## 2. Configuração do Ambiente
Instalação e download dos modelos necessários para NLTK e spaCy.

In [ ]:
# Instalação e download do modelo em Português do spaCy
!pip install -q nltk spacy scikit-learn pandas
!python -m spacy download pt_core_news_sm

import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import spacy
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Recursos do NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

print("✅ Ambiente configurado com sucesso!")

---
## 3. Definição do Corpus de Exemplo
Vamos trabalhar com frases em português sobre NLP e dados.

In [ ]:
corpus = [
    "O processamento de linguagem natural analisa textos e linguagem humana.",
    "Modelos de aprendizado de máquina processam textos e dados.",
    "A linguagem humana é rica em textos e contextos."
]

for i, doc in enumerate(corpus, 1):
    print(f"Documento {i}: {doc}")

---
## 4. Prática com NLTK + Scikit-Learn

Usamos o **NLTK** para pré-processamento (tokenização, remoção de stopwords e pontuações) e o **scikit-learn** para gerar as matrizes.

In [ ]:
stop_words_nltk = set(stopwords.words('portuguese'))

def preprocess_nltk(text):
    tokens = word_tokenize(text.lower())
    # Filtra palavras alfanuméricas removendo stopwords
    filtered = [t for t in tokens if t.isalnum() and t not in stop_words_nltk]
    return " ".join(filtered)

corpus_nltk = [preprocess_nltk(doc) for doc in corpus]

print("--- Corpus após NLTK ---")
for i, doc in enumerate(corpus_nltk, 1):
    print(f"Doc {i}: {doc}")

### 4.1 Bag of Words (BoW) com NLTK

In [ ]:
bow_vec = CountVectorizer()
bow_matrix = bow_vec.fit_transform(corpus_nltk)

df_bow_nltk = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_vec.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)

print("Matriz Bag of Words (Contagens):")
display(df_bow_nltk)

### 4.2 TF-IDF com NLTK

In [ ]:
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(corpus_nltk)

df_tfidf_nltk = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vec.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)

print("Matriz TF-IDF (Pesos normalizados):")
display(df_tfidf_nltk.round(3))

---
## 5. Prática com spaCy (Pipeline + Lematização)

O **spaCy** permite fazer **lematização** (converter verbos conjugados e plurais para o lema raiz, unificando termos como *processamento* / *processam* e *texto* / *textos*).

In [ ]:
nlp = spacy.load("pt_core_news_sm")

def preprocess_spacy(text):
    doc = nlp(text.lower())
    # Remove pontuação, stopwords e extrai o lemma de cada token
    lemmas = [
        token.lemma_
        for token in doc
        if not token.is_stop and not token.is_punct and token.is_alpha
    ]
    return " ".join(lemmas)

corpus_spacy = [preprocess_spacy(doc) for doc in corpus]

print("--- Corpus após spaCy (Lematizado) ---")
for i, doc in enumerate(corpus_spacy, 1):
    print(f"Doc {i}: {doc}")

### 5.1 TF-IDF com termos Lematizados pelo spaCy

In [ ]:
tfidf_spacy_vec = TfidfVectorizer()
tfidf_spacy_mat = tfidf_spacy_vec.fit_transform(corpus_spacy)

df_tfidf_spacy = pd.DataFrame(
    tfidf_spacy_mat.toarray(),
    columns=tfidf_spacy_vec.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(corpus))]
)

print("Matriz TF-IDF com spaCy (Lematizado):")
display(df_tfidf_spacy.round(3))

---
## 6. Comparativo e Resumo Final

| Abordagem | Vantagem Principal | Desvantagem |
| :--- | :--- | :--- |
| **BoW (NLTK)** | Simples, intuitivo e rápido | Sensível a stopwords e palavras frequentes |
| **TF-IDF (NLTK)** | Destaca palavras raras e informativas | Não unifica variações morfológicas da mesma palavra |
| **TF-IDF (spaCy)** | Reduz dimensionalidade via lematização (*texto/textos* viram um só) | Exige pipeline linguístico mais pesado |